# Notebook 02 — EDA ponderado por factor de expansión

Exploración del dataset que dejé limpio en el notebook 01. Cinco preguntas concretas:

1. ¿Cómo se distribuyen las horas trabajadas por sexo? (porque ahí está el bulto de la brecha)
2. ¿Cómo se distribuye el ingreso mensual por sexo?
3. ¿Cómo cambia la brecha mensual según nivel educativo?
4. ¿Cómo varía la brecha entre entidades federativas?
5. ¿Cuál es el perfil promedio de hombres vs. mujeres ocupadas en variables Mincer?

Todo ponderado por `fac_tri`. Los gráficos quedan en `outputs/` listos para el README.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src import enoe_loader as enoe

plt.rcParams.update({
    "figure.dpi": 100, "savefig.dpi": 160, "savefig.bbox": "tight",
    "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 11, "axes.titlesize": 13, "axes.titleweight": "bold",
})

COLOR_H = "#2c7fb8"  # azul para hombres
COLOR_M = "#d95f0e"  # naranja para mujeres
OUTPUTS = ROOT / "outputs"
OUTPUTS.mkdir(exist_ok=True)

df = pd.read_parquet(ROOT / "data" / "processed" / "enoen_2025_4t_mincer.parquet")
h = df[df["mujer"] == 0]
m = df[df["mujer"] == 1]
print(f"Dataset: {len(df):,} registros · {df['fac_tri'].sum():,.0f} representados")
print(f"Hombres: {len(h):,} | Mujeres: {len(m):,}")


## Tabla resumen: caracterización ponderada por sexo

Antes de los gráficos, una tabla con las medias ponderadas de las variables clave. La columna `Brecha %` se calcula como `(H - M) / H * 100` — positiva cuando los hombres están "arriba", negativa cuando lo están las mujeres.


In [ ]:
def resumen(grp, w):
    return {
        "n": len(grp),
        "representa": int(w.sum()),
        "edad": enoe.media_ponderada(grp["edad"], w),
        "escolaridad": enoe.media_ponderada(grp["anios_escolaridad"], w),
        "experiencia": enoe.media_ponderada(grp["experiencia"], w),
        "horas/sem": enoe.media_ponderada(grp["horas_semanales"], w),
        "ingreso_mensual": enoe.media_ponderada(grp["ingreso_mensual_w"], w),
        "ingreso_hora": enoe.media_ponderada(grp["ingreso_hora_w"], w),
        "log_ingreso_mensual": enoe.media_ponderada(grp["log_ingreso_mensual_w"], w),
        "% formal": enoe.media_ponderada(grp["formal"].astype(float), w) * 100,
    }

tabla = pd.DataFrame({"Hombres": resumen(h, h["fac_tri"]),
                       "Mujeres": resumen(m, m["fac_tri"])})
tabla["Brecha %"] = ((tabla["Hombres"] - tabla["Mujeres"]) / tabla["Hombres"] * 100).round(2)
tabla.round(2)


**Lo que la tabla revela:**

Hombres y mujeres ocupadas en México tienen edad y experiencia casi idénticas. Las mujeres tienen **más** años de escolaridad (11.0 vs 10.3) y están **más** en el sector formal (57.8% vs 55.1%). El único bloque donde hay diferencia material es **horas trabajadas** — los hombres trabajan 8.4 hrs más por semana en promedio.

Esto contradice dos clichés habituales: "las mujeres ganan menos porque están menos calificadas" (falso — están más calificadas) y "las mujeres ganan menos porque están en la informalidad" (falso — están más formalizadas).


## G1 — Distribución de horas semanales por sexo

El histograma ponderado del corazón del proyecto.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
bins = np.arange(0, 80, 2)
ax.hist(h["horas_semanales"], bins=bins, weights=h["fac_tri"]/h["fac_tri"].sum(),
        alpha=0.55, color=COLOR_H, label="Hombres")
ax.hist(m["horas_semanales"], bins=bins, weights=m["fac_tri"]/m["fac_tri"].sum(),
        alpha=0.55, color=COLOR_M, label="Mujeres")

hrs_h = enoe.media_ponderada(h["horas_semanales"], h["fac_tri"])
hrs_m = enoe.media_ponderada(m["horas_semanales"], m["fac_tri"])
ax.axvline(hrs_h, color=COLOR_H, linestyle="--", linewidth=1.5)
ax.axvline(hrs_m, color=COLOR_M, linestyle="--", linewidth=1.5)
ax.annotate(f"{hrs_h:.1f} hrs", xy=(hrs_h, 0.08), xytext=(hrs_h+7, 0.085),
            color=COLOR_H, fontsize=10, fontweight="bold",
            arrowprops=dict(arrowstyle="-", color=COLOR_H))
ax.annotate(f"{hrs_m:.1f} hrs", xy=(hrs_m, 0.10), xytext=(hrs_m-15, 0.105),
            color=COLOR_M, fontsize=10, fontweight="bold",
            arrowprops=dict(arrowstyle="-", color=COLOR_M))

ax.set_xlabel("Horas semanales trabajadas")
ax.set_ylabel("Proporción de personas ocupadas")
ax.set_title(f"Las mujeres trabajan {hrs_h-hrs_m:.1f} hrs menos por semana en el mercado remunerado")
ax.text(0.99, -0.18, "ENOE 4T 2025 — INEGI. Ponderado por factor de expansión.",
        transform=ax.transAxes, ha="right", fontsize=8, color="gray")
ax.legend(loc="upper right")
plt.savefig(OUTPUTS / "eda_g1_horas.png")
plt.show()


## G2 — Distribución del log del ingreso mensual

La distribución que va a modelar Mincer. El eje x está en escala log pero etiquetado en pesos para que se entienda.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
bins = np.linspace(6, 12, 60)
ax.hist(h["log_ingreso_mensual_w"], bins=bins, weights=h["fac_tri"]/h["fac_tri"].sum(),
        alpha=0.55, color=COLOR_H, label="Hombres")
ax.hist(m["log_ingreso_mensual_w"], bins=bins, weights=m["fac_tri"]/m["fac_tri"].sum(),
        alpha=0.55, color=COLOR_M, label="Mujeres")

log_h = enoe.media_ponderada(h["log_ingreso_mensual_w"], h["fac_tri"])
log_m = enoe.media_ponderada(m["log_ingreso_mensual_w"], m["fac_tri"])
ax.axvline(log_h, color=COLOR_H, linestyle="--", linewidth=1.5)
ax.axvline(log_m, color=COLOR_M, linestyle="--", linewidth=1.5)

ticks = np.log([2000, 5000, 10000, 20000, 50000])
ax.set_xticks(ticks)
ax.set_xticklabels([f"${int(np.exp(t)):,}" for t in ticks])
ax.set_xlim(6.5, 11.5)
ax.set_xlabel("Ingreso mensual (escala logarítmica)")
ax.set_ylabel("Proporción")
ax.set_title("La distribución de las mujeres está corrida a la izquierda en todo el rango")
ax.text(0.99, -0.18, "ENOE 4T 2025 — winsorizado p1–p99 y ponderado por fac_tri.",
        transform=ax.transAxes, ha="right", fontsize=8, color="gray")
ax.legend(loc="upper right")
plt.savefig(OUTPUTS / "eda_g2_ingreso_mensual.png")
plt.show()


## G3 — Brecha por nivel educativo

¿La brecha es más grande arriba (techo de cristal) o abajo (piso pegajoso)?


In [ ]:
# nivel_esc ya está en el parquet, pero por si acaso lo reconstruyo
if "nivel_esc" not in df.columns:
    df["nivel_esc"] = pd.cut(df["anios_escolaridad"],
                              bins=[-0.01, 6, 9, 12, 16, 24],
                              labels=["Primaria","Secundaria","Media","Licenciatura","Posgrado"])

filas = []
for nivel, grp in df.groupby("nivel_esc", observed=True):
    hh = grp[grp["mujer"]==0]; mm = grp[grp["mujer"]==1]
    ing_h = enoe.media_ponderada(hh["ingreso_mensual_w"], hh["fac_tri"])
    ing_m = enoe.media_ponderada(mm["ingreso_mensual_w"], mm["fac_tri"])
    hrs_h = enoe.media_ponderada(hh["horas_semanales"], hh["fac_tri"])
    hrs_m = enoe.media_ponderada(mm["horas_semanales"], mm["fac_tri"])
    filas.append({"nivel": nivel,
                   "brecha_mensual": (ing_h-ing_m)/ing_h*100,
                   "brecha_horas":   (hrs_h-hrs_m)/hrs_h*100})
brecha_esc = pd.DataFrame(filas).set_index("nivel")
brecha_esc.round(1)


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(brecha_esc))
ax.bar(x - 0.2, brecha_esc["brecha_horas"], width=0.4, color="#7fcdbb", label="Brecha en horas (%)")
ax.bar(x + 0.2, brecha_esc["brecha_mensual"], width=0.4, color="#41b6c4", label="Brecha mensual (%)")
for i, (bh, bm) in enumerate(zip(brecha_esc["brecha_horas"], brecha_esc["brecha_mensual"])):
    ax.text(i-0.2, bh+0.4, f"{bh:.1f}%", ha="center", fontsize=9, color="#1d6680")
    ax.text(i+0.2, bm+0.4, f"{bm:.1f}%", ha="center", fontsize=9, color="#1d6680")
ax.set_xticks(x)
ax.set_xticklabels(brecha_esc.index)
ax.set_ylabel("Brecha (%)")
ax.set_title("La brecha mensual cae con el nivel educativo, pero no desaparece")
ax.text(0.99, -0.18, "ENOE 4T 2025 — niveles, ponderado.",
        transform=ax.transAxes, ha="right", fontsize=8, color="gray")
ax.legend(loc="upper right")
plt.savefig(OUTPUTS / "eda_g3_brecha_por_educacion.png")
plt.show()


**Lo que la gráfica revela:** La brecha mensual baja de 33% en primaria a 18% en posgrado. La brecha en horas sigue el mismo patrón. Pero la brecha mensual incluso en posgrado sigue siendo de 18 puntos — la educación reduce la brecha pero no la cierra.


## G4 — Brecha mensual por entidad federativa

¿La brecha es uniforme en todo el país o se concentra en algunos estados? Mapa a vista de pájaro antes del modelo.


In [ ]:
ESTADOS = {
    1:"Aguascalientes", 2:"Baja California", 3:"Baja California Sur", 4:"Campeche",
    5:"Coahuila", 6:"Colima", 7:"Chiapas", 8:"Chihuahua", 9:"Ciudad de México",
    10:"Durango", 11:"Guanajuato", 12:"Guerrero", 13:"Hidalgo", 14:"Jalisco",
    15:"Estado de México", 16:"Michoacán", 17:"Morelos", 18:"Nayarit", 19:"Nuevo León",
    20:"Oaxaca", 21:"Puebla", 22:"Querétaro", 23:"Quintana Roo", 24:"San Luis Potosí",
    25:"Sinaloa", 26:"Sonora", 27:"Tabasco", 28:"Tamaulipas", 29:"Tlaxcala",
    30:"Veracruz", 31:"Yucatán", 32:"Zacatecas",
}

ent_filas = []
for ent_id, grp in df.groupby("ent", observed=True):
    hh = grp[grp["mujer"]==0]; mm = grp[grp["mujer"]==1]
    if len(hh) < 30 or len(mm) < 30: continue
    ing_h = enoe.media_ponderada(hh["ingreso_mensual_w"], hh["fac_tri"])
    ing_m = enoe.media_ponderada(mm["ingreso_mensual_w"], mm["fac_tri"])
    ent_filas.append({"entidad": ESTADOS.get(int(ent_id), str(ent_id)),
                       "brecha_mensual": (ing_h-ing_m)/ing_h*100})
brecha_ent = pd.DataFrame(ent_filas).sort_values("brecha_mensual").reset_index(drop=True)
prom = brecha_ent["brecha_mensual"].mean()

fig, ax = plt.subplots(figsize=(10, 9))
colores = ["#41b6c4" if v < prom else "#1d6680" for v in brecha_ent["brecha_mensual"]]
ax.barh(brecha_ent["entidad"], brecha_ent["brecha_mensual"], color=colores, alpha=0.9)
for i, v in enumerate(brecha_ent["brecha_mensual"]):
    ax.text(v + 0.4, i, f"{v:.1f}%", va="center", fontsize=8.5)
ax.axvline(prom, color="gray", linestyle="--", alpha=0.7)
ax.text(prom+0.3, -1.6, f"Promedio nacional: {prom:.1f}%", color="gray", fontsize=9)
ax.set_xlabel("Brecha mensual de ingreso (%)")
ax.set_title("La brecha varía 4x entre entidades — Chiapas 8%, Hidalgo 32%")
ax.text(0.99, -0.04, "ENOE 4T 2025 — entidades con ≥30 registros por sexo.",
        transform=ax.transAxes, ha="right", fontsize=8, color="gray")
plt.savefig(OUTPUTS / "eda_g4_brecha_por_entidad.png")
plt.show()


**Aviso de cautela:** Chiapas saliendo con la brecha más baja (8%) es contraintuitivo. Probablemente es sesgo de selección: en Chiapas la participación laboral femenina es muy baja, así que las mujeres que sí están ocupadas son una muestra muy filtrada — típicamente con más educación que el promedio nacional. Esto es exactamente por qué el notebook 05 va a hacer corrección Heckman. Por ahora lo dejo flagueado y no lo presento como hallazgo robusto.


## G5 — Caracterización Mincer por sexo

Las cuatro variables que van a entrar a la regresión Mincer, con las medias ponderadas por sexo lado a lado. Es la imagen que sostiene la narrativa del proyecto.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7.5))
axes = axes.flatten()

casos = [
    ("Escolaridad (años)",  "anios_escolaridad", 12, "+{:.1f} años a favor de mujeres"),
    ("Experiencia (años)",  "experiencia",       27, "Prácticamente iguales"),
    ("% en sector formal",  "formal",            65, "+{:.1f} pp a favor de mujeres"),
    ("Horas semanales",     "horas_semanales",   52, "−{:.1f} hrs en mujeres — única diferencia grande"),
]

for ax, (titulo, col, xmax, plantilla) in zip(axes, casos):
    if col == "formal":
        vh = enoe.media_ponderada(h[col].astype(float), h["fac_tri"]) * 100
        vm = enoe.media_ponderada(m[col].astype(float), m["fac_tri"]) * 100
        sufijo = "%"
    else:
        vh = enoe.media_ponderada(h[col], h["fac_tri"])
        vm = enoe.media_ponderada(m[col], m["fac_tri"])
        sufijo = ""
    ax.barh(["Hombres","Mujeres"], [vh, vm], color=[COLOR_H, COLOR_M], alpha=0.85)
    ax.set_xlim(0, xmax)
    diff = abs(vm - vh)
    es_neg = (col == "horas_semanales")
    sub = plantilla.format(diff)
    color_titulo = "#b32b1f" if es_neg else "black"
    ax.set_title(f"{titulo}\n{sub}", loc="left", fontsize=11, color=color_titulo)
    ax.text(vh*0.97, 0, f"{vh:.1f}{sufijo}", va="center", ha="right", color="white", fontweight="bold")
    ax.text(vm*0.97, 1, f"{vm:.1f}{sufijo}", va="center", ha="right", color="white", fontweight="bold")
    ax.set_xticks([])

fig.suptitle("Las mujeres ocupadas mexicanas tienen más educación y más formalidad.\n"
             "La única variable Mincer donde están en desventaja son las horas trabajadas.",
             fontsize=12.5, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(OUTPUTS / "eda_g5_caracterizacion_mincer.png")
plt.show()


## Debrief del notebook 02

**Lo que dejó claro el EDA:**

1. La brecha mensual de 23% NO se explica por menor calificación femenina. Las mujeres ocupadas en México tienen más años de escolaridad y están más en el sector formal que los hombres.
2. La única dimensión Mincer donde hay diferencia material es horas trabajadas — 8.4 hrs/semana menos. Eso por sí solo explica gran parte de la brecha mensual.
3. La brecha mensual baja con el nivel educativo (33% en primaria → 18% en posgrado) pero no se cierra. El "techo" educativo reduce pero no elimina.
4. Hay variación geográfica fuerte (8% en Chiapas, 32% en Hidalgo). Chiapas requiere lectura cautelosa por sesgo de selección — pendiente para Heckman.

**Lo que se me resistió:**

El intento inicial de usar `nombre_entidad` (columna `n_ent` en SDEMT) salió mal — esa columna trae códigos pequeños, no nombres de estado. Tuve que mapear `ent` (1-32) contra el catálogo INEGI a mano. Lección: nunca asumas qué hay detrás de un nombre de columna sin imprimirlo primero.

**Para el notebook 03 (regresión Mincer):**

Modelar `log_ingreso_mensual_w ~ log_horas_w + anios_escolaridad + experiencia + experiencia2 + C(sector) + formal + C(ent)`, separado por sexo (WLS con peso `fac_tri`). Reportar coeficientes con errores robustos HC3. Verificar que el retorno a la escolaridad esté entre 8% y 12% (rango típico de la literatura mexicana) como sanity check.
